In [ ]:
import numpy as np
import pandas as pd

months_per_cycle = 132
overlap_months = 20
n_cycles = 4

cycle_strengths = [80, 120, 150, 100]

phase_lag_months = 36

total_months = months_per_cycle * n_cycles

rows = []

for month in range(1, total_months + 1):

    for cycle in range(n_cycles):

        cycle_start = cycle * months_per_cycle - overlap_months

        if month < cycle_start or month > cycle_start + months_per_cycle:
            continue

        S_n = cycle_strengths[cycle]

        phase_n = (month - cycle_start) / months_per_cycle
        phase_s = (month - cycle_start - phase_lag_months) / months_per_cycle

        cycle_sign = (-1)**cycle
        Tn = 1.73 - 0.0035 * S_n

        # ----------------------
        # NORTH
        # ----------------------
        if 0 <= phase_n <= 1:

            x = phase_n

            lambda_n = 12.2 + 0.022 * S_n
            lambda_i = (26.4 - 34.2*x + 16.1*x**2) * (lambda_n / 14.6)
            sigma_i = (0.14 + 1.05*x - 0.78*x**2) * lambda_i

            R_n = S_n * np.sin(np.pi * x)**2
            N_pairs_n = int(R_n / 2.75 / 2)

            for s in range(N_pairs_n):

                lat = np.random.normal(lambda_i, sigma_i)

                # --- NEW: allow some cross-equatorial emergence ---
                if np.random.rand() < 0.1:
                    lat = np.random.normal(0, 8)

                lon = np.random.uniform(0, 360)

                A = np.random.lognormal(mean=np.log(45), sigma=np.log(3))

                area_m2 = A * 3.05e16
                radius_m = np.sqrt(area_m2 / np.pi)
                radius_Mm = (radius_m / 1e6) / 100

                flux = (7e19 * A) / 1e17

                tilt = cycle_sign * Tn * np.sqrt(abs(lat))

                # --- NEW: tilt noise (break symmetry) ---
                tilt += np.random.normal(0, 8)

                rows.append([month, lat, tilt, radius_Mm, lon, flux])


        # ----------------------
        # SOUTH (ASYMMETRIC)
        # ----------------------
        if 0 <= phase_s <= 1:

            x = phase_s

            lambda_n = 12.2 + 0.022 * S_n
            lambda_i = (26.4 - 34.2*x + 16.1*x**2) * (lambda_n / 14.6)

            # --- NEW: slightly different latitude spread ---
            sigma_i = 1.2 * (0.14 + 1.05*x - 0.78*x**2) * lambda_i

            # --- NEW: stronger asymmetry in activity ---
            R_s = 0.6 * S_n * np.sin(np.pi * x)**2
            N_pairs_s = int(R_s / 2.75 / 2)

            for s in range(N_pairs_s):

                lat = -np.random.normal(lambda_i, sigma_i)

                # --- cross-equatorial emergence ---
                if np.random.rand() < 0.15:
                    lat = np.random.normal(0, 10)

                lon = np.random.uniform(0, 360)

                A = np.random.lognormal(mean=np.log(45), sigma=np.log(3))

                area_m2 = A * 3.05e16
                radius_m = np.sqrt(area_m2 / np.pi)
                radius_Mm = (radius_m / 1e6) / 100

                flux = (7e19 * A) / 1e17

                # --- NEW: break Joy's law symmetry ---
                tilt = -0.7 * cycle_sign * Tn * np.sqrt(abs(lat))

                # --- add noise ---
                tilt += np.random.normal(0, 10)

                rows.append([month, lat, tilt, radius_Mm, lon, flux])


# Save file
df = pd.DataFrame(rows)
df.to_csv("BMR_4cycles_phase_lag_4.txt", sep=" ", header=False, index=False, float_format="%.6f")

print("Total months:", total_months)
print("Total BMRs:", len(df))

Total months: 528
Total BMRs: 8104
